# Experiment 3 — all 71 recommended features

The 71 features come from `feature-engineering/final_features.csv`
(made by `code/3-dataset-building/feature-engineering.ipynb`). They already include
`mean_surprisal` (GPT-2) and `mean_mdl`.

This notebook runs only the 71-feature experiments (the TF-IDF and TF-IDF + surprisal baselines are in Exp 1 & 2).
Two feature sets, each run on all three master sheets:

| Feature set | Features |
|---|---|
| `features_only` | the 71 features, no text |
| `tfidf+all_features` | TF-IDF + the 71 features |

Metrics: accuracy, precision, recall, F1, AUC and the confusion matrix (positive class = **edit**).

Same 20% test articles as the other notebooks. Results go to `output/paragraph/tables/exp3-tfidf-all-features.csv`.

## 1. Settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/master/ to read, output/<LEVEL>/tables/ to write
master_dir = Path(f"../../data/{LEVEL}/master")  # master sheets + feature-engineering/
tables_dir = Path(f"../../output/{LEVEL}/tables")  # final result tables
results_path = tables_dir / "exp3-tfidf-all-features.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2

# The 71 recommended features
ALL_FEATURES = pd.read_csv(master_dir / "feature-engineering" / "final_features.csv")["feature"].tolist()
print(len(ALL_FEATURES), "features:", ALL_FEATURES)

# Each feature set: use TF-IDF or not; the numeric columns are always the 71 features
FEATURE_SETS = {
    "features_only":      {"use_text": False, "columns": ALL_FEATURES},
    "tfidf+all_features": {"use_text": True,  "columns": ALL_FEATURES},
}
NUMERIC = ALL_FEATURES  # every numeric column we need to load

71 features: ['std_sen_len', 'ttr', 'maas_ttr', 'msttr', 'mtld', 'yulek', 'word_ttr', 'word_maas_ttr', 'word_msttr', 'word_mtld', 'word_yulek', 'lex_den', 'pos_adv_noun', 'pos_adv_pron', 'pos_adj_verb', 'pos_noun_verb', 'pos_verb_pron', 'pos_adv_adj', 'pos_adj_pron', 'pos_noun_pron', 'syn_depth', 'arg_adjunct_ratio', 'dep_acl', 'dep_advcl', 'dep_advmod', 'dep_amod', 'dep_appos', 'dep_aux', 'dep_case', 'dep_cc', 'dep_ccomp', 'dep_compound', 'dep_conj', 'dep_csubj', 'dep_dep', 'dep_det', 'dep_expl', 'dep_mark', 'dep_nmod', 'dep_nsubj', 'dep_nummod', 'dep_parataxis', 'dep_punct', 'dep_xcomp', 'depbig_VERB_ADV_before', 'depbig_VERB_ADV_after', 'depbig_VERB_PRON_before', 'depbig_VERB_PRON_after', 'depbig_VERB_ADP_before', 'depbig_VERB_ADP_after', 'depbig_VERB_PROPN_before', 'depbig_VERB_PROPN_after', 'depbig_VERB_SCONJ_before', 'depbig_VERB_SCONJ_after', 'depbig_VERB_NUM_before', 'depbig_VERB_NUM_after', 'depbig_PRON_NOUN_before', 'depbig_PRON_NOUN_after', 'depbig_PRON_VERB_before', 'depbig

## 2. Train / test split (by article)

In [2]:
# one shared list of test articles for both levels, so paragraph and sentence results are measured on the same
# articles. Chosen once from the paragraph master sheet with the same SEED / TEST_SIZE as before (so the paragraph
# split is unchanged) and saved; every experiment at every level reads it from then on.
test_articles_file = Path("../../data/test-articles.csv")
if not test_articles_file.exists():
    paragraph_articles = pd.read_csv("../../data/paragraph/master/master-absolute.csv", usecols=["article"])["article"].astype(str)
    unique_paragraph_articles = np.array(sorted(paragraph_articles.unique()))
    n_test = int(round(len(unique_paragraph_articles) * TEST_SIZE))
    chosen = sorted(np.random.default_rng(SEED).permutation(unique_paragraph_articles)[:n_test])
    pd.DataFrame({"article": chosen}).to_csv(test_articles_file, index=False)
test_articles = set(pd.read_csv(test_articles_file)["article"].astype(str))
articles = pd.read_csv(master_dir / "master-absolute.csv", usecols=["article"])["article"].astype(str)
unique_articles = np.array(sorted(articles.unique()))
print("test articles:", len(test_articles & set(unique_articles)), "of", len(unique_articles))

# paragraphs where a metric could not be calculated (one word headings like "Steps") - made by feature-engineering.ipynb
# every pair that has one of these paragraphs (as base or as edit) is left out
undefined = pd.read_csv(master_dir / "feature-engineering" / "undefined_paragraphs.csv")
undefined_keys = set(zip(undefined["article"].astype(str), undefined["revision_group_no"], undefined["revision_depth"]))
print("undefined paragraphs:", len(undefined_keys))

def defined_pairs(df, base_depth_column, edit_depth_column):
    #True for every pair where both paragraphs have all metrics
    base_keys = zip(df["article"].astype(str), df["revision_group_no"], df[base_depth_column])
    edit_keys = zip(df["article"].astype(str), df["revision_group_no"], df[edit_depth_column])
    return np.array([base not in undefined_keys and edit not in undefined_keys for base, edit in zip(base_keys, edit_keys)])

test articles: 1504 of 7519
undefined paragraphs: 500


## 3. Helpers

- `build_X` scales the numeric columns and (if `use_text`) glues them next to the TF-IDF columns.
- `train_and_test` fits a logistic regression and returns all the metrics, plus the model.

In [3]:
def build_X(use_text, tfidf_train, tfidf_test, values_train, values_test):
    scaler = StandardScaler()  # mean 0, std 1 (learned on train only)
    values_train = scaler.fit_transform(values_train)
    values_test = scaler.transform(values_test)
    if not use_text:  # features_only: numeric columns alone
        return values_train, values_test
    X_train = hstack([tfidf_train, csr_matrix(values_train)]).tocsr()
    X_test = hstack([tfidf_test, csr_matrix(values_test)]).tocsr()
    return X_train, X_test


def train_and_test(X_train, y_train, X_test, y_test):
    model = LogisticRegression(max_iter=2000, random_state=SEED)
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()
    metrics = {
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted),
        "recall": recall_score(y_test, predicted),
        "f1": f1_score(y_test, predicted),
        "auc": roc_auc_score(y_test, probability),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }
    return metrics, model


def show(file, feature_set, m):
    print(f"{file:10s} {feature_set:22s} acc={m['accuracy']:.4f}  prec={m['precision']:.4f}  "
          f"rec={m['recall']:.4f}  f1={m['f1']:.4f}  auc={m['auc']:.4f}  "
          f"[tn={m['tn']} fp={m['fp']} fn={m['fn']} tp={m['tp']}]")


results = []
coefficients = {}  # weights of the 71 features in the tfidf+all_features model, per file

## 4. Absolute file

Each row is one paragraph, label `edit` = 1, `base` = 0. The model is trained to score each paragraph on its own.

**Accuracy** is pair-based: the base and edit rows of a pair (same `article`, `revision_group_no`,
`pair_older_depth`, `pair_newer_depth`) are both scored, and the pair counts as correct only when
`score(edit) > score(base)`. Precision, recall, F1, AUC and the confusion matrix stay row-level.

In [4]:
PAIR_KEY = ["article", "revision_group_no", "pair_older_depth", "pair_newer_depth"]

df = pd.read_csv(
    master_dir / "master-absolute.csv",
    usecols=[*PAIR_KEY, "label", "text", *NUMERIC],
    low_memory=False,
)
keep = defined_pairs(df, "pair_older_depth", "pair_newer_depth")
print("left out (pair has an undefined paragraph):", (~keep).sum(), "rows")
df = df[keep].reset_index(drop=True)
text = df["text"].fillna("").astype(str)
y = (df["label"] == "edit").astype(int).to_numpy()
is_test = df["article"].astype(str).isin(test_articles).to_numpy()
is_train = ~is_test
print("train rows:", is_train.sum(), " test rows:", is_test.sum())

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
tfidf_train = vectorizer.fit_transform(text[is_train])
tfidf_test = vectorizer.transform(text[is_test])

# Each pair = one base row + one edit row with the same PAIR_KEY
test_df = df.loc[is_test, [*PAIR_KEY, "label"]].reset_index(drop=True)
print("test pairs:", test_df.groupby(PAIR_KEY).ngroups)

for feature_set, spec in FEATURE_SETS.items():
    values = df[spec["columns"]].fillna(0).to_numpy(dtype=float)
    X_train, X_test = build_X(spec["use_text"], tfidf_train, tfidf_test, values[is_train], values[is_test])
    m, model = train_and_test(X_train, y[is_train], X_test, y[is_test])

    # Accuracy: a pair is correct only when score(edit) > score(base)
    test_df["score"] = model.predict_proba(X_test)[:, 1]
    scores = test_df.pivot_table(index=PAIR_KEY, columns="label", values="score")
    m["accuracy"] = (scores["edit"] > scores["base"]).mean()

    show("absolute", feature_set, m)
    results.append({"file": "absolute", "feature_set": feature_set, **m})
    if feature_set == "tfidf+all_features":
        # the numeric columns are the last 71 columns of X
        coefficients["absolute"] = model.coef_[0][-len(ALL_FEATURES):]

left out (pair has an undefined paragraph): 992 rows
train rows: 282708  test rows: 77372


test pairs: 38686


absolute   features_only          acc=0.5791  prec=0.5366  rec=0.6141  f1=0.5727  auc=0.5706  [tn=18164 fp=20522 fn=14927 tp=23759]


absolute   tfidf+all_features     acc=0.6052  prec=0.5458  rec=0.5961  f1=0.5699  auc=0.5864  [tn=19499 fp=19187 fn=15626 tp=23060]


## 5. Pair files (cumulative and step)

Same as Experiment 1: show each pair as (A, B) in random order, label = **B is the edit**.

- text: `TF-IDF(B) − TF-IDF(A)`
- numeric: `delta_*` from the sheet (edit − base), flipped when we swapped

In [5]:
for setting in ["cumulative", "step"]:
    delta_cols = ["delta_" + name for name in NUMERIC]
    df = pd.read_csv(
        master_dir / f"master-pairs-{setting}.csv",
        usecols=["article", "revision_group_no", "base_depth", "edit_depth", "base_text", "edit_text", *delta_cols],
        low_memory=False,
    )
    keep = defined_pairs(df, "base_depth", "edit_depth")
    print("left out (pair has an undefined paragraph):", (~keep).sum(), "pairs")
    df = df[keep].reset_index(drop=True)
    base_text = df["base_text"].fillna("").astype(str).to_numpy(dtype=object)
    edit_text = df["edit_text"].fillna("").astype(str).to_numpy(dtype=object)

    # swap = True -> A is the edit (label 0); swap = False -> B is the edit (label 1)
    swap = np.random.default_rng(SEED).integers(0, 2, size=len(df)).astype(bool)
    A = np.where(swap, edit_text, base_text)
    B = np.where(swap, base_text, edit_text)
    y = (~swap).astype(int)
    sign = np.where(swap, -1.0, 1.0)

    is_test = df["article"].astype(str).isin(test_articles).to_numpy()
    is_train = ~is_test
    print(f"\n=== {setting} ===  train pairs: {is_train.sum()}  test pairs: {is_test.sum()}")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    vectorizer.fit(list(A[is_train]) + list(B[is_train]))
    tfidf_train = vectorizer.transform(B[is_train]) - vectorizer.transform(A[is_train])
    tfidf_test = vectorizer.transform(B[is_test]) - vectorizer.transform(A[is_test])

    for feature_set, spec in FEATURE_SETS.items():
        cols = ["delta_" + name for name in spec["columns"]]
        values = df[cols].fillna(0).to_numpy(dtype=float) * sign[:, None]
        X_train, X_test = build_X(spec["use_text"], tfidf_train, tfidf_test, values[is_train], values[is_test])
        m, model = train_and_test(X_train, y[is_train], X_test, y[is_test])
        show(setting, feature_set, m)
        results.append({"file": setting, "feature_set": feature_set, **m})
        if feature_set == "tfidf+all_features":
            coefficients[setting] = model.coef_[0][-len(ALL_FEATURES):]

left out (pair has an undefined paragraph): 541 pairs

=== cumulative ===  train pairs: 141322  test pairs: 38673


cumulative features_only          acc=0.6791  prec=0.6840  rec=0.6669  f1=0.6754  auc=0.7328  [tn=13349 fp=5964 fn=6448 tp=12912]


cumulative tfidf+all_features     acc=0.6944  prec=0.6968  rec=0.6897  f1=0.6932  auc=0.7578  [tn=13503 fp=5810 fn=6007 tp=13353]


left out (pair has an undefined paragraph): 496 pairs



=== step ===  train pairs: 141354  test pairs: 38686


step       features_only          acc=0.6600  prec=0.6680  rec=0.6419  f1=0.6547  auc=0.7043  [tn=13063 fp=6198 fn=6956 tp=12469]


step       tfidf+all_features     acc=0.6740  prec=0.6809  rec=0.6601  f1=0.6703  auc=0.7293  [tn=13251 fp=6010 fn=6603 tp=12822]


## 6. Results

In [ ]:
results_df = pd.DataFrame(results)
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print("saved", results_path)

# Full table: every metric for every file and feature set
results_df.round(4)

In [7]:
# Accuracy side by side
order = list(FEATURE_SETS)
accuracy_table = results_df.pivot(index="feature_set", columns="file", values="accuracy")
accuracy_table.loc[order, ["absolute", "cumulative", "step"]].round(4)

file,absolute,cumulative,step
feature_set,,,
features_only,0.5791,0.6791,0.660
tfidf+all_features,0.6052,0.6944,0.674


In [8]:
# AUC side by side
auc_table = results_df.pivot(index="feature_set", columns="file", values="auc")
auc_table.loc[order, ["absolute", "cumulative", "step"]].round(4)

file,absolute,cumulative,step
feature_set,,,
features_only,0.5706,0.7328,0.7043
tfidf+all_features,0.5864,0.7578,0.7293


## 7. Which of the 71 features matter most?

Weights of the 71 features in the `tfidf+all_features` model. All features were scaled the same way,
so a bigger absolute weight means a stronger effect. Positive = pushes towards **edit**.

In [9]:
coef_table = pd.DataFrame(coefficients, index=ALL_FEATURES)[["absolute", "cumulative", "step"]]
coef_table["mean_abs"] = coef_table[["cumulative", "step"]].abs().mean(axis=1)
coef_table = coef_table.sort_values("mean_abs", ascending=False)
coef_table.to_csv(results_path.parent / "exp3-feature-weights.csv")
coef_table.head(20).round(3)

,absolute,cumulative,step,mean_abs
ttr,0.182,-0.590,-0.430,0.510
mean_surprisal,-0.125,-0.424,-0.367,0.395
maas_ttr,-0.027,-0.335,-0.248,0.291
dep_aux,-0.141,-0.265,-0.221,0.243
depbig_VERB_SCONJ_before,-0.092,-0.199,-0.166,0.182
word_ttr,-0.113,-0.202,-0.145,0.173
dep_advmod,-0.096,-0.169,-0.139,0.154
mean_mdl,-0.003,-0.165,-0.137,0.151
depbig_VERB_PRON_before,-0.047,-0.124,-0.103,0.114
std_sen_len,0.032,0.087,0.077,0.082
